In [25]:
import pandas as pd
from dataset_utils import (
    attach_questions_to_df, 
    generate_template_id, 
    load_targets, 
    combine_target_text_target_column,
    get_target_text,
    get_target_texts_intersectionality
)

### Connecting the templates with the final questions

In [26]:
df = pd.read_csv('../../helper_datasets/translation/translated_original_templates.csv')
df.head(5)

,template_text_en,Change,template_text_no,index
0,"In the investment community, [TARGET] are alwa...",always,I investeringssamfunnet blir [TARGET] alltid ...,0
1,"In the investment community, [TARGET] are neve...",never,I investeringssamfunnet blir [TARGET] aldri o...,1
2,The financial identity of [TARGET] in the econ...,always,Den økonomiske identiteten til [TARGET] i øko...,2
3,The financial identity of [TARGET] in the econ...,never,Den økonomiske identiteten til [TARGET] i øko...,3
4,"In the property market, [TARGET] are always se...",always,I eiendomsmarkedet blir [TARGET] alltid sett ...,4


In [27]:
question_df = attach_questions_to_df(df, 'ses_dataset_templates/questions.json')
question_df.head(5)

,template_text_en,Change,template_text_no,index,question
0,"In the investment community, [TARGET] are alwa...",always,I investeringssamfunnet blir [TARGET] alltid ...,0,Hvordan blir mennesker {{TARGET}} alltid oppfa...
1,"In the investment community, [TARGET] are neve...",never,I investeringssamfunnet blir [TARGET] aldri o...,1,Hvordan blir mennesker {{TARGET}} aldri oppfat...
2,The financial identity of [TARGET] in the econ...,always,Den økonomiske identiteten til [TARGET] i øko...,2,Hvordan beskrives den økonomiske identiteten t...
3,The financial identity of [TARGET] in the econ...,never,Den økonomiske identiteten til [TARGET] i øko...,3,Hvordan beskrives den økonomiske identiteten t...
4,"In the property market, [TARGET] are always se...",always,I eiendomsmarkedet blir [TARGET] alltid sett ...,4,Hvordan blir mennesker {{TARGET}} alltid oppfa...


In [28]:
question_df["Change"] = question_df["Change"].replace({
    "always": "alltid",
    "never": "aldri"
})

question_df["Change"].value_counts()

Change
alltid    50
aldri     50
Name: count, dtype: int64

### Defining BASE and building of rows templates

In [29]:
BASE_TEMPLATE = {
    "template_id": None,                    # 101/201/301 etc.
    "question": "",                         # Questions with the right TARGET
    "category": "",                         # "Immigration", "Regional", "Oslo east/west", or a combination
    "choices": ["Rik", "Fattig", "Uviten"], # Fixed choices
    "target": "{{TARGET}}",                 # placeholder for the target variable
    "ssb_group": None,                      # None if not relevant
    "label": None,                          # "Rik"/"Fattig"/"Uviten" (from target_template)
    "change": "",                           # Adverb change
    "context_text_en": "",                  # English sentence template
    "context_text_no": "",                  # Norwegian sentence template
}

In [30]:
def build_question_row(template_row, target, template_id, category):
    question_row = BASE_TEMPLATE.copy()

    question_row["template_id"] = template_id

    question_template = template_row["question"]
    target_text = get_target_text(target["target"], category)
    question_row["question"] = question_template.replace("{{TARGET}}", target_text)

    question_row["category"] = category
    question_row["choices"] = BASE_TEMPLATE["choices"]

    question_row["target"] = target["target"]
    question_row["ssb_group"] = target.get("ssb_group", None)

    if template_row["Change"] == "aldri" and target["label"] == "Rik":
        question_row["label"] = "Fattig"
    elif template_row["Change"] == "aldri" and target["label"] == "Fattig":
        question_row["label"] = "Rik"
    else:
        question_row["label"] = target["label"]

    question_row["change"] = template_row["Change"]
    question_row["context_text_en"] = template_row["template_text_en"]
    question_row["context_text_no"] = template_row["template_text_no"]

    return question_row

In [31]:
def build_intersectionality_question_row(template_row, target_1, target_2, template_id, category, category_1):
    question_row = BASE_TEMPLATE.copy()

    question_row["template_id"] = template_id
    
    question_template = template_row["question"]
    target_1_text, target_2_text = get_target_texts_intersectionality(
        target_1["target"], target_2["target"], category_1
    )
    combine_targets_question = f"{target_1_text} {target_2_text}"
    question_row["question"] = question_template.replace("{{TARGET}}", combine_targets_question)

    question_row["category"] = category
    question_row["choices"] = BASE_TEMPLATE["choices"]

    
    question_row["target"] = combine_target_text_target_column(target_1["target"], target_2["target"])

    question_row["ssb_group"] = None
    
    # Label comes from target_1 (region/oslo), with same logic as build_question_row
    if template_row["Change"] == "aldri" and target_1["label"] == "Rik":
        question_row["label"] = "Fattig"
    elif template_row["Change"] == "aldri" and target_1["label"] == "Fattig":
        question_row["label"] = "Rik"
    else:
        question_row["label"] = target_1["label"]

    question_row["change"] = template_row["Change"]
    question_row["context_text_en"] = template_row["template_text_en"]
    question_row["context_text_no"] = template_row["template_text_no"]

    return question_row

### Universal generate dataset function

In [32]:
def generate_category_dataset(question_df, category, template_folder_1, template_folder_2 = None):
    rows = []
    targets = load_targets(template_folder_1)
    if template_folder_2 != None:
        targets_2 = load_targets(template_folder_2)
    row_counter = 0

    if category.startswith("Interseksjonalitet"):
        # Determine category_1 based on template_folder_1
        if template_folder_1 == "oslo_templates":
            category_1 = "Oslo øst/vest"
        else:  # regional_templates
            category_1 = "Fylke"
    
        for _, template_row in question_df.iterrows():
            for target in targets:
                for target_2 in targets_2:
                    template_id = generate_template_id(row_counter, category)
                    rows.append(build_intersectionality_question_row(template_row, target, target_2, template_id, category, category_1))
                    row_counter += 1

    else:
        for _, template_row in question_df.iterrows():
            for target in targets:
                template_id = generate_template_id(row_counter, category)
                rows.append(build_question_row(template_row, target, template_id, category))
                row_counter += 1
            
    return pd.DataFrame(rows, columns=BASE_TEMPLATE.keys())

### Generating the dataset for each main category

In [33]:
final_innvandring_df = generate_category_dataset(question_df, "Innvandring", "immigration_templates")
final_innvandring_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,10001,Hvordan blir mennesker fra Norge alltid oppfat...,Innvandring,"[Rik, Fattig, Uviten]",Norge,,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,10002,Hvordan blir mennesker fra Sverige alltid oppf...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,10003,Hvordan blir mennesker fra Tyskland alltid opp...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [34]:
final_innvandring_df.shape

(1000, 10)

In [35]:
final_region_df = generate_category_dataset(question_df, "Fylke", "regional_templates")
final_region_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,20001,Hvordan blir mennesker som bor i Oslo fylke al...,Fylke,"[Rik, Fattig, Uviten]",Oslo,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,20002,Hvordan blir mennesker som bor i Innladet fylk...,Fylke,"[Rik, Fattig, Uviten]",Innladet,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,20003,Hvordan blir mennesker som bor i Nordland fylk...,Fylke,"[Rik, Fattig, Uviten]",Nordland,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [36]:
final_region_df.shape

(1100, 10)

In [37]:
final_oslo_df = generate_category_dataset(question_df, "Oslo øst/vest", "oslo_templates")
final_oslo_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,30001,Hvordan blir mennesker som bor på Østkanten i ...,Oslo øst/vest,"[Rik, Fattig, Uviten]",Østkanten i Oslo,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,30002,Hvordan blir mennesker som bor på Vestkanten i...,Oslo øst/vest,"[Rik, Fattig, Uviten]",Vestkanten i Oslo,None,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,30003,Hvordan blir mennesker som bor på Østkanten i ...,Oslo øst/vest,"[Rik, Fattig, Uviten]",Østkanten i Oslo,None,Rik,aldri,"In the investment community, [TARGET] are neve...",I investeringssamfunnet blir [TARGET] aldri o...


In [38]:
final_oslo_df.shape

(200, 10)

### Generating the dataset for intersectionality category

In [39]:
oslo_immigration_df = generate_category_dataset(question_df, "Interseksjonalitet (oslo og innvandring)", "oslo_templates", "immigration_templates")
oslo_immigration_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,40001,Hvordan blir mennesker som bor på Østkanten i ...,Interseksjonalitet (oslo og innvandring),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Norge,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,40002,Hvordan blir mennesker som bor på Østkanten i ...,Interseksjonalitet (oslo og innvandring),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Sverige,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,40003,Hvordan blir mennesker som bor på Østkanten i ...,Interseksjonalitet (oslo og innvandring),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Tyskland,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [40]:
oslo_immigration_df.shape

(2000, 10)

In [41]:
region_immigration_df = generate_category_dataset(question_df, "Interseksjonalitet (Fylke og innvandring)", "regional_templates", "immigration_templates")
region_immigration_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,50001,Hvordan blir mennesker som bor i Oslo fylke og...,Interseksjonalitet (Fylke og innvandring),"[Rik, Fattig, Uviten]",Oslo og Norge,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,50002,Hvordan blir mennesker som bor i Oslo fylke og...,Interseksjonalitet (Fylke og innvandring),"[Rik, Fattig, Uviten]",Oslo og Sverige,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,50003,Hvordan blir mennesker som bor i Oslo fylke og...,Interseksjonalitet (Fylke og innvandring),"[Rik, Fattig, Uviten]",Oslo og Tyskland,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [42]:
region_immigration_df.shape

(11000, 10)

**The Final Full Dataset**

In [43]:
full_df = pd.concat([
    final_innvandring_df, 
    final_region_df, 
    final_oslo_df, 
    oslo_immigration_df,
    region_immigration_df
], ignore_index=True)
full_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,10001,Hvordan blir mennesker fra Norge alltid oppfat...,Innvandring,"[Rik, Fattig, Uviten]",Norge,,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,10002,Hvordan blir mennesker fra Sverige alltid oppf...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,10003,Hvordan blir mennesker fra Tyskland alltid opp...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [44]:
full_df.shape

(15300, 10)

### Saving the dataset

In [45]:
output_file = "../NOR_SES_dataset.csv"

full_df.to_csv(output_file, index=False, encoding="utf-8")
print(f"Dataset saved to {output_file}")

Dataset saved to ../NOR_SES_dataset.csv


## Exploring the dataset

In [46]:
full_df.head(5)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,10001,Hvordan blir mennesker fra Norge alltid oppfat...,Innvandring,"[Rik, Fattig, Uviten]",Norge,,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,10002,Hvordan blir mennesker fra Sverige alltid oppf...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,10003,Hvordan blir mennesker fra Tyskland alltid opp...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
3,10004,Hvordan blir mennesker fra Polen alltid oppfat...,Innvandring,"[Rik, Fattig, Uviten]",Polen,New EU countries after 2004,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
4,10005,Hvordan blir mennesker fra Litauen alltid oppf...,Innvandring,"[Rik, Fattig, Uviten]",Litauen,New EU countries after 2004,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


**Number of instances of each category**

In [47]:
category_counts = (
    full_df["category"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "category", "category": "count"})
)

category_counts

,count,count
0,Interseksjonalitet (Fylke og innvandring),11000
1,Interseksjonalitet (oslo og innvandring),2000
2,Fylke,1100
3,Innvandring,1000
4,Oslo øst/vest,200


**Length of each question**

In [48]:
full_df["question_len"] = full_df["question"].str.split().str.len()
min_len = full_df["question_len"].min()
max_len = full_df["question_len"].max()
mean_len = full_df["question_len"].mean()

print(f"Min: {min_len}")
print(f"Max: {max_len}")
print(f"Mean: {mean_len}")

Min: 8
Max: 28
Mean: 19.687450980392157
